<a href="https://colab.research.google.com/github/GR0409/Statistical-Learning-e21091/blob/main/Assignment_7b_Gaussian_Mixture_Model_Clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q. Bayesian Estimation of a User Ability Parameter from Item Responses

An online learning platform presents a user with a sequence of $n$ multiple-choice questions **one at a time**. Each question is either answered correctly or incorrectly, allowing the platform to update its estimate of the user's ability dynamically after every response.

Let $Y_i$ denote the user's response to the $i$-th item encountered:

$$Y_i=
\begin{cases}
1, & \text{if the user answers item } i \text{ correctly},\\
0, & \text{if the user answers item } i \text{ incorrectly}.
\end{cases}$$

The platform assumes that the probability of a correct response is governed by a two-parameter logistic (2PL) item response model. Specifically, conditional on the user's latent ability parameter $\Theta=\theta$, the response probability for item $i$ is:

$$P(Y_i=1\mid \Theta=\theta)=p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}},$$

where $a_i>0$ is the known discrimination parameter, and $b_i$ is the known difficulty parameter of item $i$.

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed responses** up to the current step $k$ (where $1 \le k \le n$).

Before observing any responses, the platform initializes the user's latent ability estimate with a standard normal prior distribution:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{\theta^2}{2}\right) \quad \text{implying} \quad \Theta \sim \mathscr{N}(0,1).$$

As the user progresses, the posterior distribution at step $k-1$ serves as the prior distribution for step $k$.

---

### Tasks

1. **Visualizing the Mechanics:** Plot $P(Y_i=1\mid \Theta=\theta)$ vs $\theta$ using Plotly for two distinct values of $a_i$, where one of those $a_i$ values is paired with three different difficulty values of $b_i$. Interpret how moving $b_i$ shifts the curve horizontally.
2. **Sequential Likelihood Contribution:** Write down the likelihood contribution $L(y_k \mid \theta)$ of a *single* new response $y_k$ at step $k$, given the latent ability $\theta$. Then, write down the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.
3. **Mathematical Formulation of the Running Update:** Write down the recursive relationship for the posterior density at step $k$, denoted $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$, up to a proportionality constant, using the prior state $f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)})$ and the new observation $y_k$.
4. **Dynamic Shifting:** Explain how a correct answer ($y_k = 1$) to a highly difficult item (large $b_k$) mathematically shifts the peak of the running posterior density distribution relative to the previous step.
5. **Tracking Certainty and Sharpness:** Explain how the discrimination parameter $a_k$ of the current item alters the variance (or "sharpness") of the distribution during a running update. What happens when $a_k$ is very large versus very small?
6. **Numerical Implementation of a Running Grid:** Describe a algorithmic approach to numerically approximate and maintain this running posterior density function on a fixed grid of $\theta$-values. Explicitly state how you would perform the sequential normalization step computationally after an item is answered.


7. **Evaluating Convergence over the Timeline:** Suppose the user's true, hidden latent ability is $\theta_{\text{true}} = 0.75$. Write a Python script that extends your previous grid simulation to track the performance of the running estimators over a sequence of $n = 20$ items.
* **Simulate Responses:** Dynamically generate the user's responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against the true response probability $p_k(\theta_{\text{true}})$. Give each item a random difficulty $b_k \sim \mathscr{N}(0, 1)$ and a random discrimination $a_k \sim \text{Uniform}(0.5, 2.0)$.
* **Track Estimators:** At each step $k$, calculate and store the running Posterior Mean ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$) and the running Maximum A Posteriori ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$) estimate.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $20$. Add a static horizontal reference line at $y = 0.75$ representing $\theta_{\text{true}}$.
* **Analysis:** Briefly explain how the distance between your estimators and $\theta_{\text{true}}$ changes as $k$ increases, and interpret what this implies about the platform's confidence in its measurement.


# Answers

1.

In [ ]:
import numpy as np
import plotly.graph_objects as go

theta = np.linspace(-4, 4, 100)
def p_i(theta, a, b): return 1 / (1 + np.exp(-a * (theta - b)))

fig = go.Figure()
# a = 1.0, varying b
for b in [-1, 0, 1]:
    fig.add_trace(go.Scatter(x=theta, y=p_i(theta, 1.0, b), name=f"a=1.0, b={b}"))
# a = 2.0, varying b
for b in [-1, 0, 1]:
    fig.add_trace(go.Scatter(x=theta, y=p_i(theta, 2.0, b), name=f"a=2.0, b={b}", line=dict(dash='dash')))

fig.update_layout(title="2PL IRT Model Response Probabilities", xaxis_title="Theta", yaxis_title="P(Y=1 | Theta)")
fig.show()

2.

The likelihood of a single new response $y_k$ is a Bernoulli mass function,

$$L(y_k \mid \theta) = [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$

The joint likelihood for the running history vector $\mathbf{y}^{(k)}$ is the product of independent contributions,

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{j=1}^k \left[ \frac{1}{1+e^{-a_j(\theta-b_j)}} \right]^{y_j} \left[ 1 - \frac{1}{1+e^{-a_j(\theta-b_j)}} \right]^{1 - y_j}$$

3.

Using Bayes' theorem, the recursive relationship drops the marginal likelihood denominator,

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \cdot [p_k(\theta)]^{y_k} [1 - p_k(\theta)]^{1 - y_k}$$

4.

If $y_k = 1$ for a large $b_k$, the likelihood $p_k(\theta)$ is near $0$ for low $\theta$ and approaches $1$ for high $\theta$. Multiplying the prior density by this monotonically increasing likelihood heavily penalizes the density at low $\theta$ values, mathematically pulling the peak of the posterior density to the right.

5.

The discrimination parameter $a_k$ dictates the steepness of the logistic likelihood curve.  

Very large $a_k$ - The likelihood resembles a step function, aggressively chopping off density below $b_k$. This drastically reduces posterior variance, creating a sharper density.

Very small $a_k$ - The likelihood is nearly flat. Multiplication by a flat function preserves the prior's shape, leaving the variance and sharpness largely unchanged.

6.

Numerical Implementation of a Running GridDefine a discrete, equally spaced array of $\theta$ values. Initialize the prior density array using the standard normal PDF evaluated at the grid points.  Upon observing $y_k$, evaluate the likelihood $L(y_k \mid \theta)$ at every grid point.Multiply the prior array by the likelihood array element wise to get the unnormalized posterior.

Sequential Normalization - Integrate the unnormalized array using the trapezoidal rule. Divide every element in the unnormalized array by this scalar integral to ensure the density sums to 1.

7.



In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

np.random.seed(42)
n = 20
theta_true = 0.75
theta_grid = np.linspace(-5, 5, 1000)
prior = norm.pdf(theta_grid, 0, 1)

a_vals = np.random.uniform(0.5, 2.0, n)
b_vals = np.random.normal(0, 1, n)

mean_estimates = []
map_estimates = []

for k in range(n):
    # Simulate response
    p_true = 1 / (1 + np.exp(-a_vals[k] * (theta_true - b_vals[k])))
    y_k = 1 if np.random.uniform(0, 1) < p_true else 0

    # Likelihood and Update
    p_grid = 1 / (1 + np.exp(-a_vals[k] * (theta_grid - b_vals[k])))
    likelihood = (p_grid**y_k) * ((1 - p_grid)**(1 - y_k))
    unnormalized = prior * likelihood

    # Normalize
    prior = unnormalized / np.trapezoid(unnormalized, theta_grid)

    # Estimates
    mean_est = np.trapezoid(theta_grid * prior, theta_grid)
    map_est = theta_grid[np.argmax(prior)]

    mean_estimates.append(mean_est)
    map_estimates.append(map_est)

# Plotting
fig = go.Figure()
fig.add_trace(go.Scatter(y=mean_estimates, mode='lines+markers', name='Posterior Mean'))
fig.add_trace(go.Scatter(y=map_estimates, mode='lines+markers', name='MAP'))
fig.add_hline(y=theta_true, line_dash="dash", line_color="red", annotation_text="True Theta")
fig.update_layout(title="Sequential Estimators over 20 Items", xaxis_title="Step k", yaxis_title="Estimate")
fig.show()

# Q. Bayesian Tracking of Click-Through Rates (CTR) via Conjugate Beta-Binomial Updates

An e-commerce platform wants to optimize its recommendation engine by dynamically estimating the click-through rate (CTR) of a newly launched advertisement. Since user traffic arrives continuously, the platform updates its belief about the advertisement's performance **one impression at a time** rather than waiting for large batch updates.

Let $\Theta = \theta$ represent the true, hidden conversion rate (probability of a click) of the advertisement, where $\theta \in [0, 1]$.

Let $Y_k$ denote a single user's interaction with the advertisement at time step $k$:

$$Y_k =
\begin{cases}
1, & \text{if the user clicks the advertisement}, \\
0, & \text{if the user does not click the advertisement}.
\end{cases}$$

The platform assumes that conditional on the true conversion rate $\Theta = \theta$, each user interaction is an independent Bernoulli trial:

$$P(Y_k = 1 \mid \Theta = \theta) = \theta$$

Let $\mathbf{y}^{(k)} = (y_1, y_2, \dots, y_k)$ represent the **running vector of observed user interactions** up to the current impression step $k$ (where $1 \le k \le n$).

Before observing any data, the platform assigns a flexible **Beta distribution** as the initial prior over the unknown parameter $\Theta$:

$$f_{\Theta}^{(0)}(\theta) = \frac{1}{\mathrm{B}(\alpha_0, \beta_0)} \theta^{\alpha_0 - 1} (1 - \theta)^{\beta_0 - 1} \quad \text{implying} \quad \Theta \sim \text{Beta}(\alpha_0, \beta_0)$$

where $\mathrm{B}(\cdot, \cdot)$ is the Beta function acting as the normalizing constant. Under a sequential framework, the posterior distribution at step $k-1$ serves directly as the prior distribution for step $k$.

---

**Tasks**

**1. Structural Probability and Properties**
Plot the probability density function (PDF) of a $\text{Beta}(\alpha, \beta)$ distribution using Plotly for three distinct parameter pairs:

* Uninformative state: $(\alpha=1, \beta=1)$
* Right-skewed state: $(\alpha=2, \beta=8)$
* Left-skewed state: $(\alpha=8, \beta=2)$

Interpret how changing the balance between $\alpha$ and $\beta$ shifts the center of mass of the density function over the domain $[0, 1]$.

**2. Sequential Likelihood and Joint History**

Write down the mathematical likelihood contribution $L(y_k \mid \theta)$ of a *single* isolated response $y_k$ at step $k$, given the click probability $\theta$. Following this, express the joint likelihood function for the running history vector $\mathbf{y}^{(k)}$.

**3. Closed-Form Analytical Updates (Conjugacy)**

Using Bayes' Theorem, derive the recursive algebraic relationship for the posterior density at step $k$, denoted as $f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)})$. Prove analytically that the posterior remains in the Beta family (**Beta-Binomial Conjugacy**) by explicitly writing down the closed-form update parameters $\alpha_k$ and $\beta_k$ as simple arithmetic updates of $\alpha_{k-1}$, $\beta_{k-1}$, and $y_k$. Also compute the **Posterior Mean** of the latent parameter $\Theta$ at time step $k$ (i.e. $\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}]$).


**4. Dynamic Shifting Mechanics**

Explain how an observed click ($y_k = 1$) vs. a non-click ($y_k = 0$) shifts the peak of the running density distribution mathematically. Contrast this analytical framework against non-conjugate setups (such as the 2PL IRT model) where numerical grid integration is strictly required.

**5. Running Point Estimators**

State the exact closed-form equations used to evaluate the following point estimates at step $k$ directly from the updated shape parameters $\alpha_k$ and $\beta_k$:

* **Running Posterior Mean** ($\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$)
* **Running Maximum A Posteriori** ($\widehat{\theta}_{\mathrm{MAP}}^{(k)}$)

**6. Performance Tracking and Convergence Analysis**

Suppose the advertisement's true, hidden click-through rate is $\theta_{\text{true}} = 0.35$. Write a Python script to track the performance of your closed-form sequential estimators over a timeline of $n = 100$ impressions:

* **Initialize State:** Set the base prior parameters to $\alpha_0 = 1, \beta_0 = 1$ (representing uniform initial uncertainty).
* **Simulate Responses:** Dynamically generate user responses $y_k \in \{0, 1\}$ at each step by comparing a random draw from a Uniform distribution $U(0,1)$ against $\theta_{\text{true}}$.
* **Track Estimators:** Loop through each step, update $\alpha_k$ and $\beta_k$ analytically, and store the computed values for $\widehat{\theta}_{\mathrm{Bayes}}^{(k)}$ and $\widehat{\theta}_{\mathrm{MAP}}^{(k)}$.
* **Visualize:** Use Plotly to create a single line chart showing the progression of both estimators from step $0$ to $100$. Add a static horizontal reference line at $y = 0.35$ representing $\theta_{\text{true}}$.
* **Analysis:** Explain how the distance between your estimators and $\theta_{\text{true}}$ responds as the sampling size $k$ approaches $100$. What does this imply about the accumulation of evidence over time relative to the choice of the initial prior?

# Answers

1.


In [ ]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

x = np.linspace(0, 1, 500)
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=beta.pdf(x, 1, 1), name="Uninformative (1,1)"))
fig.add_trace(go.Scatter(x=x, y=beta.pdf(x, 2, 8), name="Right-skewed (2,8)"))
fig.add_trace(go.Scatter(x=x, y=beta.pdf(x, 8, 2), name="Left-skewed (8,2)"))
fig.update_layout(title="Beta Distribution PDFs", xaxis_title="Theta", yaxis_title="Density")
fig.show()

2.

Sequential Likelihood and Joint History Single response likelihood,

$$L(y_k \mid \theta) = \theta^{y_k} (1 - \theta)^{1 - y_k}$$

Joint likelihood for the history,

$$L(\mathbf{y}^{(k)} \mid \theta) = \prod_{j=1}^k \theta^{y_j} (1 - \theta)^{1 - y_j} = \theta^{\sum_{j=1}^k y_j} (1 - \theta)^{k - \sum_{j=1}^k y_j}$$

3.

Closed-Form Analytical Updates (Conjugacy)Using Bayes' Theorem,

$$f_{\Theta \mid \mathbf{Y}^{(k)}}(\theta \mid \mathbf{y}^{(k)}) \propto f_{\Theta \mid \mathbf{Y}^{(k-1)}}(\theta \mid \mathbf{y}^{(k-1)}) \cdot L(y_k \mid \theta)$$

$$\propto \left[ \theta^{\alpha_{k-1} - 1} (1 - \theta)^{\beta_{k-1} - 1} \right] \cdot \left[ \theta^{y_k} (1 - \theta)^{1 - y_k} \right]$$

$$= \theta^{(\alpha_{k-1} + y_k) - 1} (1 - \theta)^{(\beta_{k-1} + 1 - y_k) - 1}$$
This matches the exact structural form of a Beta density. Therefore, the updates are closed-form arithmetic,

$$\alpha_k = \alpha_{k-1} + y_k$$

$$\beta_k = \beta_{k-1} + (1 - y_k)$$

The Posterior Mean at step $k$ is:

$$\mathbb{E}[\Theta \mid \mathbf{Y}^{(k)}=\mathbf{y}^{(k)}] = \frac{\alpha_k}{\alpha_k + \beta_k}$$

4.

Dynamic Shifting Mechanics
An observed click ($y_k=1$) adds 1 to $\alpha$, increasing the numerator of the expected value and mathematically shifting the peak to the right. A non-click ($y_k=0$) adds 1 to $\beta$, increasing the denominator and shifting the peak left. Unlike the 2PL IRT model which requires computational grid arrays and integrals, conjugate updates require $O(1)$ arithmetic addition with no numerical approximation.  

5.

Running Point EstimatorsRunning Posterior Mean,

$\widehat{\theta}_{\mathrm{Bayes}}^{(k)} = \frac{\alpha_k}{\alpha_k + \beta_k}$  

Running MAP,

$\widehat{\theta}_{\mathrm{MAP}}^{(k)} = \frac{\alpha_k - 1}{\alpha_k + \beta_k - 2}$ (for $\alpha_k, \beta_k > 1$)

7.

In [ ]:
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)
n = 100
theta_true = 0.35
alpha, beta = 1, 1

mean_ests, map_ests = [], []

for k in range(n):
    y_k = 1 if np.random.uniform(0, 1) < theta_true else 0
    alpha += y_k
    beta += (1 - y_k)

    mean_ests.append(alpha / (alpha + beta))
    # MAP bounded to avoid negative/invalid parameters in early stages
    map_val = (alpha - 1) / (alpha + beta - 2) if (alpha + beta) > 2 else alpha / (alpha + beta)
    map_ests.append(map_val)

fig = go.Figure()
fig.add_trace(go.Scatter(y=mean_ests, mode='lines', name='Posterior Mean'))
fig.add_trace(go.Scatter(y=map_ests, mode='lines', name='MAP'))
fig.add_hline(y=theta_true, line_dash="dash", line_color="red", annotation_text="True Theta")
fig.update_layout(title="Beta-Binomial Estimates over 100 Impressions", xaxis_title="Step k")
fig.show()